## Creating Spark Session

In [87]:
import os
import sys

In [88]:
# 1. Install Dependencies
!pip install -q pyspark findspark openml pyarrow

In [89]:
#This searches your system for an Apache Spark installation and configures your Python environment to recognize where the Spark libraries reside.
import findspark
findspark.init()

In [90]:
#SparkSession: This is the universal entry point for programming Spark with the Dataset and DataFrame AP
from pyspark.sql import SparkSession 
#These are specialized, highly optimized functions that execute on distributed Spark data nodes
from pyspark.sql.functions import (
    col, count, when, isnan, row_number, rank, dense_rank, 
    lag, lead, broadcast, expr, lit, current_timestamp, avg, sum, desc
)
#from pyspark.sql.window import Window
from pyspark.sql.window import Window
#Pipeline: A sequence of data preprocessing steps and algorithms chained together to automate and organize a machine learning workflow.
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler
import pandas as pd
import openml

spark = SparkSession.builder \
    .appName("Enterprise_Pipeline") \
    .master("local[*]") \
    .config("spark.driver.memory", "4g") \
    .config("spark.sql.shuffle.partitions", "8") \
    .config("spark.sql.execution.arrow.pyspark.enabled", "true") \
    .getOrCreate()


In [91]:
#Create local folder for keep data output
for folder in ['data/bronze', 'data/silver', 'data/gold']:
    os.makedirs(folder, exist_ok=True)

## Reading Data 

In [92]:
df = spark.read.csv(
    "BNPParibas_Data.csv",
    header=True,
    inferSchema=True
)


## Display :
Schema

Record Count

Null 

Duplicate Count

Data Types


In [93]:
df.printSchema()

root
 |-- customer_id: integer (nullable = true)
 |-- age: integer (nullable = true)
 |-- tenure_months: integer (nullable = true)
 |-- monthly_charges: double (nullable = true)
 |-- total_charges: double (nullable = true)
 |-- contract_type: string (nullable = true)
 |-- internet_service: string (nullable = true)
 |-- support_tickets: integer (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- churn: integer (nullable = true)



Record Count


In [94]:
df.count()

1000

In [95]:
df.describe().show()

+-------+-----------------+------------------+------------------+-----------------+------------------+--------------+----------------+------------------+--------------+------------------+
|summary|      customer_id|               age|     tenure_months|  monthly_charges|     total_charges| contract_type|internet_service|   support_tickets|payment_method|             churn|
+-------+-----------------+------------------+------------------+-----------------+------------------+--------------+----------------+------------------+--------------+------------------+
|  count|             1000|              1000|              1000|             1000|              1000|          1000|            1000|              1000|          1000|              1000|
|   mean|            500.5|            43.819|            35.459|79.96715000000002| 2800.235379999997|          NULL|            NULL|             1.956|          NULL|             0.502|
| stddev|288.8194360957494|14.991029650093076|20.36819044937

Null Count

In [96]:
df.select([
    count(when(col(c).isNull(),c)).alias(c)
    for c in df.columns]).show()


+-----------+---+-------------+---------------+-------------+-------------+----------------+---------------+--------------+-----+
|customer_id|age|tenure_months|monthly_charges|total_charges|contract_type|internet_service|support_tickets|payment_method|churn|
+-----------+---+-------------+---------------+-------------+-------------+----------------+---------------+--------------+-----+
|          0|  0|            0|              0|            0|            0|               0|              0|             0|    0|
+-----------+---+-------------+---------------+-------------+-------------+----------------+---------------+--------------+-----+



Duplicate Count

In [100]:
total_count=df.count()
unique_count=df.drop_duplicates().count()
duplicate_count=total_count-unique_count
duplicate_count

0

Data Types

In [98]:
df.dtypes

[('customer_id', 'int'),
 ('age', 'int'),
 ('tenure_months', 'int'),
 ('monthly_charges', 'double'),
 ('total_charges', 'double'),
 ('contract_type', 'string'),
 ('internet_service', 'string'),
 ('support_tickets', 'int'),
 ('payment_method', 'string'),
 ('churn', 'int')]